In [1]:
CREATION_SYSTEM_PROMPT="""
### ROLE & OBJECTIVE
You are a professional brochure layout engine. Your sole responsibility is to convert a provided project payload into one complete, visually rich, print-ready HTML brochure.

### OUTPUT FORMAT (STRICT)
1.  **Format:** Output raw, valid HTML5 code only.
2.  **No JSON:** Do NOT output JSON. Do NOT wrap the output in a JSON object.
3.  **Start/End:** Your response must start immediately with `<!DOCTYPE html>` and end with `</html>`.
4.  **No Chatter:** Do not include any conversational text, logs, preambles, or explanations.
5.  **One Artifact:** Output exactly one HTML document containing all CSS and content.

### CORE DATA RULES (NON-NEGOTIABLE)
1.  **Factual Integrity:** * The payload is the ONLY source of truth. 
    * Never invent, infer, estimate, or extrapolate project-specific facts.
    * Never infer facts from images.
2.  **Missing Data:** * Never output "missing", "null", "N/A", or placeholders.
    * If data is absent, use short, neutral, tasteful brochure copy (e.g., "A thoughtfully designed residential environment.").
    * If specific amenities are missing, do not generate generic ones; simply omit the section.
3.  **Sanitization:** Strip or escape all HTML from payload values.

### TECHNICAL SPECIFICATIONS
1.  **Structure:** Valid HTML5.
2.  **Styling:** * All CSS must be embedded in a `<style>` block in the `<head>`.
    * Use CSS variables for theming.
    * **Page Size:** Explicitly define `@page { size: A4 landscape; margin: 0; }` to ensure print readiness.
3.  **Pagination:** * Produces exactly 3–4 A4 landscape pages.
    * Use explicit page breaks (`break-after: page`).
    * Include Paged.js script immediately before `</body>`: `<script src="https://unpkg.com/pagedjs/dist/paged.polyfill.js"></script>`

### VISUAL & IMAGE RULES
**Global Constraint:** Use inline SVG and CSS for most visuals. 

1.  **Raster Images (JPG, PNG, WEBP):**
    * **Allowed Locations:** Page 1 (Cover) and Page 2 (Overview) ONLY.
    * **Forbidden Locations:** Page 3 and Page 4.
    * **Quantity:** Max 1 image on Page 1. Max 1 image on Page 2. (Total Max: 2).
    * **Styling:** Must be wrapped in a fixed-geometry container with `overflow: hidden` and `object-fit: contain`.
    * **Failure Mode:** If an image causes overflow or fails to load, omit it silently.

2.  **SVGs:**
    * Must be inline code.
    * Must be accessible, proportional, and non-overflowing.

### PAGE-BY-PAGE REQUIREMENTS

**Page 1 — Cover**
* **Content:** Project Name (from payload), Tagline (payload or generic aesthetic), Location (payload).
* **Visuals:** Large hero SVG illustration OR 1 payload image (if available).
* **Highlights:** 3 badges (payload highlights or neutral creative highlights).

**Page 2 — Overview**
* **Content:** Short overview text (blend facts with tasteful filler).
* **Quick Facts Table:** Factual payload values only. Omit empty rows.
* **Amenities:** Use payload amenities only. Use inline SVG icons for bullets. If none in payload, omit section.
* **Visuals:** Max 1 payload image.

**Page 3 — Floor Plans**
* **Condition:** Omit entire page if no floor plan data exists.
* **Content:** Title, Area values, Notes (payload or neutral).
* **Visuals (STRICT):** * SVG Schematic ONLY. 
    * NO raster images allowed.
    * Draw a single-block schematic if only total area is known.
    * Constraint: Max 1 floor plan rendered.
* **Disclaimer:** "Room positions are conceptual and not a real representation."

**Page 4 — Location & Pricing**
* **Location Map:** * Inline SVG only (Abstract blocks/labeled POIs derived from payload).
    * If no data: "Well-connected to key urban conveniences."
    * NO raster images allowed.
* **Pricing:** * Table if payload has pricing. 
    * Else: "Flexible pricing options available."
* **Disclaimers:** * Map disclaimer: "Positions are conceptual and not a real representation."
    * Footer (All Pages): "Conceptual brochure — illustrative only."

### FINAL EXECUTION
Generate the HTML code now, ensuring no text appears before `<!DOCTYPE html>`.
"""

In [2]:
import os
import json
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from google import genai
from google.genai import types

# =======================
# ENV & CONFIG
# =======================

load_dotenv()

CSV_PATH = "brochure_creation_data_remaining.xlsx"
OUTPUT_DIR = "Brochures_remaining_3"
LOG_FILE = "completed_projects_shhhh.log"
MODEL_ID = "gemini-3-flash-preview"
MAX_WORKERS = 40

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =======================
# LOGGING
# =======================

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)

log_lock = Lock()

# =======================
# LOAD COMPLETED PROJECT IDS
# =======================

def load_completed_projects():
    if not os.path.exists(LOG_FILE):
        return set()

    completed = set()
    with open(LOG_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if "Completed project:" in line:
                completed.add(line.split("Completed project:", 1)[1].strip())
    return completed

completed_projects = load_completed_projects()

# =======================
# GEMINI CLIENT
# =======================

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

# =======================
# CORE WORKER FUNCTION
# =======================

def process_project(idx, row):
    project_id = str(row.get("Project Id", f"row_{idx}")).strip()

    # Skip already completed
    if project_id in completed_projects:
        return f"⏭ Skipped (already processed): {project_id}"

    payload = row.to_json(orient="records", force_ascii=False)

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=[payload],
            config=types.GenerateContentConfig(
                response_mime_type="text/plain",
                temperature=1,
                system_instruction=CREATION_SYSTEM_PROMPT,
                thinking_config=types.ThinkingConfig(thinking_level="medium")
            )
        )

        if not hasattr(response, "text") or not response.text.strip():
            return f"Empty response: {project_id}"

        try:
            data = response.text
        except:
            return f"Empty response: {project_id}"

        html_code = data
        if not html_code:
            return f"Missing HTML code: {project_id}"

        
        output_path = os.path.join(OUTPUT_DIR, f"{project_id}.html")
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(html_code)

        with log_lock:
            logging.info(f"Completed project: {project_id}")

        return f"Completed: {project_id}"

    except Exception as e:
        return f"Error ({project_id}): {e}"

# =======================
# MAIN EXECUTION
# =======================

def main():
    df = pd.read_excel(CSV_PATH)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(process_project, idx, row)
            for idx, row in df.iterrows()
        ]

        for future in tqdm(as_completed(futures), total=len(futures)):
            print(future.result())

if __name__ == "__main__":
    main()


 33%|███▎      | 1/3 [00:27<00:54, 27.18s/it]

Completed: 454571


 67%|██████▋   | 2/3 [00:30<00:12, 12.91s/it]

Completed: 460603


100%|██████████| 3/3 [00:31<00:00, 10.55s/it]

Completed: 460818
